# 霍夫圓檢測.py
**電腦視覺與深度學習 › Day14 OpenCV 電腦視覺**  ·  ⚠️ 錯誤示範(有 bug,對照修正版看)

這支程式專門找畫面裡的圓形。它把上一支的輪廓,再和霍夫圓演算法找到的圓互相比對,兩邊都同意才算是圓。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/8)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week1/DAY7"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '霍夫圓檢測.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    img = np.full((480, 640, 3), 255, np.uint8)
    cv2.circle(img, (130, 140), 70, (0, 0, 255), -1)
    cv2.circle(img, (520, 360), 55, (255, 0, 0), -1)
    cv2.rectangle(img, (260, 60), (400, 200), (0, 160, 0), -1)
    cv2.fillPoly(img, [np.array([[90, 420], [230, 420], [160, 300]])], (0, 200, 255))
    cv2.fillPoly(img, [np.array([[300, 300], [430, 280], [450, 420], [320, 440]])], (200, 0, 200))
    return img
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 開鏡頭
這段和視覺那支一樣:請出 cv2 和 numpy,打開 0 號鏡頭,打不開就結束。

In [ ]:
import cv2
import numpy as np

# 1. 建立 VideoCapture 物件
cap = cv2.VideoCapture(0)

# 檢查鏡頭是否成功開啟
if not cap.isOpened():
    print("錯誤：無法開啟視訊鏡頭。")
    exit()

### 面積範圍
先定好面積的下限 500、上限 50000,太小或太大的輪廓都不考慮。

In [ ]:
print("鏡頭已開啟！按下 'q' 鍵可結束程式。")

# 定義面積區間
min_area = 500   # 最小面積限制
max_area = 50000 # 最大面積限制 

### 讀畫面並馬上檢查
注意這裡的好習慣:讀完畫面立刻檢查 ret,讀失敗就跳出。這正好修正了視覺那支程式的問題。然後複製一份畫面,用來畫結果。

### 灰階 + 高斯模糊
霍夫圓對雜訊很敏感,所以先轉灰階、再用 13 乘 13 的高斯模糊把畫面抹平一點。模糊的大小必須是奇數。

### Canny 與輪廓
在模糊後的畫面上做 Canny,門檻是 50 和 150,再找出外圈輪廓,等一下用來篩面積和算中心點。

### 霍夫圓的參數
HoughCircles 用投票的方式找圓。minDist 是兩個圓心至少要隔多遠;param1 是內部 Canny 的高門檻;param2 是投票門檻,越小越容易找到圓,但也越容易認錯;最後兩個是半徑範圍。

### ⚠️ 把圓心轉成 uint16
有找到圓才開始比對。這裡用 np.uint16 把圓心座標轉成整數。要小心:uint16 是不能有負數的整數,只能放 0 到 65535,等一下拿它做減法就會出問題。

### 每個輪廓算面積和中心
一個一個輪廓檢查:面積在範圍內,再用影像矩算出中心點 cX、cY,做法和視覺那支一樣。

### ⚠️ 比對圓心距離:藏著溢位 bug
這段算輪廓中心和每個霍夫圓心的距離,小於 15 像素就認定是圓。可是 h_x 是 uint16,cX 減 h_x 如果是負的,數字會繞一圈變成很大的數,平方後又繞回來。差距剛好 256 時,距離會算成 0,被誤判成同一個圓。修法是先用 int 把 h_x、h_y 轉回一般整數再相減。

### 畫出確認的圓
面積對、又找得到對應的霍夫圓,才算數。這裡用紅色畫輪廓、綠色小點畫圓心,再寫上編號和半徑。

### 顯示、離開、收拾
把結果顯示出來,想看邊緣圖可以把下一行的註解打開。按 q 離開,最後放掉鏡頭、關閉視窗。

In [ ]:
while True:
    # 2. 逐影格（Frame）讀取影像
    ret, frame = cap.read()
    
    # 如果讀取失敗，直接跳出迴圈
    if not ret:
        print("錯誤：無法接收影像畫面。")
        break

    img_copy = frame.copy()
    
    # 影像預處理：轉灰階 + 高斯模糊（霍夫圓對雜訊很敏感，模糊非常關鍵）
    gray_img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray_img, (13, 13), 2)
    
    # A. 進行 Canny 與輪廓檢測 (用來篩選面積)
    edges = cv2.Canny(blurred, 50, 150)
    contours, hierarchy = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # B. 進行霍夫圓檢測
    # dp=1 (解析度), minDist=50 (圓心間的最小距離), param1=Canny高閾值, param2=圓心檢測閾值(越小越容易誤判)
    # minRadius/maxRadius 可根據實際大小調整，設為 0 代表不限制
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=100, param2=35, minRadius=10, maxRadius=200)
    
    count = 0
    
    # 如果有偵測到任何霍夫圓，才開始與輪廓進行比對
    if circles is not None and len(contours) > 0:
        # 將圓的座標轉為整數
        circles = np.uint16(np.around(circles))
        
        for cnt in contours:
            # 1. 檢查面積條件
            area = cv2.contourArea(cnt)
            if min_area <= area <= max_area:
                
                # 2. 計算輪廓中心點 (Moments)
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cX = int(M["m10"] / M["m00"])
                    cY = int(M["m01"] / M["m00"])
                    
                    # 3. 驗證這個輪廓附近有沒有對應的霍夫圓 (比對圓心距離)
                    is_circle = False
                    matched_radius = 0
                    
                    for circle in circles[0, :]:
                        h_x, h_y, h_r = circle[0], circle[1], circle[2]
                        # 計算輪廓中心與霍夫圓心的歐幾里得距離
                        distance = np.sqrt((cX - h_x)**2 + (cY - h_y)**2)
                        
                        # 如果距離小於 15 像素，我們就認定這個輪廓「確實是個圓」
                        if distance < 15:
                            is_circle = True
                            matched_radius = h_r
                            break
                    
                    # 4. 條件完全滿足：在面積區間內，且是個圓 ➔ 繪製出來
                    if is_circle:
                        count += 1
                        
                        # 繪製輪廓 (紅色)
                        cv2.drawContours(img_copy, [cnt], -1, (0, 0, 255), 2)
                        
                        # 繪製圓心 (綠色點)
                        cv2.circle(img_copy, (cX, cY), 3, (0, 255, 0), -1)
                        
                        # 標註資訊
                        cv2.putText(img_copy, f"Circle #{count} R:{matched_radius}", (cX - 30, cY - 10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    # 3. 顯示影像畫面
    cv2.imshow('Webcam', img_copy)
    # 可選：想觀察邊緣可以打開下一行
    # cv2.imshow('Edges', edges)

    # 4. 按下 'q' 鍵結束
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 5. 釋放資源
cap.release()
cv2.destroyAllWindows()

## 小結
你學到用兩種方法互相驗證,也看到一個資料型別造成的隱藏 bug。下一支凸包檢測會再加上三角形、正方形的分類。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/8) 看逐句拆解、做練習。